# Assignment 5 — Encoding Amazon Reviews
**Principles of AI · Lab 4, Part 2**

### Objective
Select a suitable encoding technique for **each of the nine original columns**, apply it, and explain the choice. The methods and parameters follow the course's **sec5_answers / Lab 4 Encoding Amazon Reviews** material. Part 1 is included as the preparation needed before Part 2.

### How to use this notebook
In Google Colab, choose **Runtime → Run all**. The original CSV downloads automatically from Kaggle; no Google Drive mount, API key, or manual upload is required. Run the cells in order. English explanations precede every code cell, and comments explain the individual code lines.

The assignment also asks for **a video in which you explain every line of code**. Use the explanation cells and comments as your speaking guide; the notebook itself does not replace that recording.

### Encoding plan
| Original column | Section method | Reason |
|---|---|---|
| Reviewer Name | Frequency encoding | Many unordered names; avoid thousands of dummy columns. |
| Profile Link | Label encoding with `pd.factorize` | A compact identifier, not a meaningful numerical measurement. |
| Country | One-hot, top 10 + Other | Countries are unordered; group rare categories to limit width. |
| Review Count | Numeric with `log1p` | Already a count; compress large values rather than encode categories. |
| Rating | Ordinal encoding, 1–5 → 0–4 | Preserve the meaningful order of ratings. |
| Review Date | Year + cyclic month | Represent year and season, including the December–January boundary. |
| Review Title | TF-IDF, 20 features | Convert short free text into weighted word features. |
| Review Text | TF-IDF, 100 features | Use a larger vocabulary for longer free text. |
| Date of Experience | Year + cyclic month + days to review | Represent timing and the delay before the review. |

**Source data:** Dongre Laxman, [Amazon Reviews Dataset](https://www.kaggle.com/datasets/dongrelaxman/amazon-reviews-dataset). The CSV is fetched from its original source, not republished in this repository.

## 1. Import Libraries

**What and why:** `numpy` provides numerical functions, `pandas` handles tables, `OrdinalEncoder` preserves rating order, and `TfidfVectorizer` converts text to numbers. The remaining standard-library tools download and open the dataset ZIP. `display` renders readable notebook tables.

**Expected result:** a confirmation that imports completed. These libraries are normally available in Google Colab.

In [1]:
import numpy as np  # Numerical functions: log1p, sin, cos, and pi.
import pandas as pd  # Read and manipulate tabular data.
from sklearn.preprocessing import OrdinalEncoder  # Encode categories in a specified order.
from sklearn.feature_extraction.text import TfidfVectorizer  # Convert text to TF-IDF vectors.
from urllib.request import urlopen  # Download the original public Kaggle archive.
from io import BytesIO  # Treat downloaded bytes as an in-memory file.
from zipfile import ZipFile  # Open the dataset ZIP archive.
from hashlib import sha256  # Record the exact dataset version used in this run.
from IPython.display import display  # Show formatted DataFrames in the notebook.
print("Libraries imported successfully.")  # Confirm that setup completed.

Libraries imported successfully.


## 2. Load the Original Dataset

**What and why:** Download the actual dataset named in the assignment. Open its `Amazon_Reviews.csv` member and read it into `df`. No synthetic data or sampling is used. The checksum identifies the bytes used in this execution.

**Read the output:** check that the original schema contains nine columns. Raw names, profile links, and review text are not printed in the saved public outputs; encoding still uses all original rows.

In [2]:
dataset_url = "https://www.kaggle.com/api/v1/datasets/download/dongrelaxman/amazon-reviews-dataset"  # Original public source.
with urlopen(dataset_url, timeout=120) as response:  # Open the download and close it automatically afterward.
    archive_bytes = response.read()  # Read the real dataset archive into memory.
with ZipFile(BytesIO(archive_bytes)) as archive:  # Open the downloaded ZIP without extracting arbitrary paths.
    csv_bytes = archive.read("Amazon_Reviews.csv")  # Read only the expected CSV member.
df = pd.read_csv(BytesIO(csv_bytes))  # Convert the CSV bytes into a DataFrame.
original_columns = df.columns.tolist()  # Keep the original schema for the final coverage check.
print("CSV SHA256:", sha256(csv_bytes).hexdigest())  # Identify the exact source snapshot used.
print("Original columns:", original_columns)  # Display the source column names.
print("Original shape:", df.shape)  # Display the number of records and columns.

CSV SHA256: 6f24067bf8b5bd6d85f64ee62155bc0b3eeac76d7362e74def262e8f7dd03f48
Original columns: ['Reviewer Name', 'Profile Link', 'Country', 'Review Count', 'Review Date', 'Rating', 'Review Title', 'Review Text', 'Date of Experience']
Original shape: (21214, 9)


## 3. Inspect Records, Types, and Missing Values

**What and why:** Inspect the table before changing it, as in Part 1 of the section. `shape` gives the size, `dtypes` gives the current types, and `isnull().sum()` counts missing cells per column. `info()` provides an additional structural summary.

**Important:** text such as `Review text not found` is not automatically counted as missing. We handle it explicitly in the next cell.

In [3]:
print("Records:", df.shape[0])  # The first dimension is the number of reviews.
print("Columns:", df.shape[1])  # The second dimension is the number of source fields.
display(pd.DataFrame({"Data type": df.dtypes.astype(str), "Missing values": df.isnull().sum()}))  # Inspect each field.
print("Placeholder review texts:", df["Review Text"].eq("Review text not found").sum())  # Count text placeholders separately.
df.info()  # Summarize column types and non-missing counts.

Records: 21214
Columns: 9


,Data type,Missing values
Reviewer Name,str,0
Profile Link,str,51
Country,str,160
Review Count,str,159
Review Date,str,159
Rating,str,159
Review Title,str,159
Review Text,str,159
Date of Experience,str,267


Placeholder review texts: 630
<class 'pandas.DataFrame'>
RangeIndex: 21214 entries, 0 to 21213
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Reviewer Name       21214 non-null  str  
 1   Profile Link        21163 non-null  str  
 2   Country             21054 non-null  str  
 3   Review Count        21055 non-null  str  
 4   Review Date         21055 non-null  str  
 5   Rating              21055 non-null  str  
 6   Review Title        21055 non-null  str  
 7   Review Text         21055 non-null  str  
 8   Date of Experience  20947 non-null  str  
dtypes: str(9)
memory usage: 1.5 MB


## 4. Clean and Prepare the Data

**What and why:** Follow the section's preparation order: remove exact duplicates, convert types, then handle missing values. Extract the rating digit and review count from their text labels. Convert both date columns to UTC datetimes; `errors="coerce"` marks invalid dates as missing instead of stopping execution.

**Missing-value decisions:**
- Drop records missing Rating or Review Date rather than inventing those values.
- Replace missing names, profile links, and countries with `Unknown`.
- Replace missing titles/text with an empty string so TF-IDF can process them.
- Fill missing Review Count with its median, which is less sensitive to large counts.
- Use Review Date for missing Date of Experience, explicitly following the section's assumption.

**Small implementation safeguard:** `.copy()` and resetting the index avoid chained-assignment and alignment problems. Assertions check the rating range and valid counts. These checks do not introduce a new encoding method.

In [4]:
before = len(df)  # Save the original row count.
df = df.drop_duplicates().copy()  # Keep one copy of each exact record and make an editable table.
print("Duplicates removed:", before - len(df))  # Report how many exact duplicates were removed.
df["Rating"] = pd.to_numeric(df["Rating"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")  # Extract the star value.
df["Review Count"] = pd.to_numeric(df["Review Count"].astype(str).str.replace(",", "").str.extract(r"(\d+)")[0], errors="coerce")  # Convert counts such as '9 reviews'.
df["Review Date"] = pd.to_datetime(df["Review Date"], errors="coerce", utc=True)  # Parse review timestamps.
df["Date of Experience"] = pd.to_datetime(df["Date of Experience"], errors="coerce", utc=True)  # Parse experience dates.
df["Review Text"] = df["Review Text"].replace("Review text not found", np.nan)  # Mark the placeholder as missing.
rows_before_drop = len(df)  # Track removal caused by missing essential fields separately.
df = df.dropna(subset=["Rating", "Review Date"]).copy()  # Remove rows missing the required rating or review date.
print("Missing-essential-field rows removed:", rows_before_drop - len(df))  # Show this second cleaning decision.
df["Country"] = df["Country"].fillna("Unknown")  # Keep missing countries as an explicit category.
df["Reviewer Name"] = df["Reviewer Name"].fillna("Unknown")  # Keep missing names as an explicit category.
df["Profile Link"] = df["Profile Link"].fillna("Unknown")  # Keep missing identifiers as an explicit category.
df["Review Title"] = df["Review Title"].fillna("")  # Empty titles can produce zero TF-IDF vectors.
df["Review Text"] = df["Review Text"].fillna("")  # Empty review texts can produce zero TF-IDF vectors.
df["Review Count"] = df["Review Count"].fillna(df["Review Count"].median())  # Impute missing counts with the median.
df["Date of Experience"] = df["Date of Experience"].fillna(df["Review Date"])  # Apply the section's date-imputation assumption.
df = df.reset_index(drop=True)  # Give all following encoding blocks the same sequential row index.
assert df["Rating"].isin([1, 2, 3, 4, 5]).all()  # Reject unexpected ratings before ordinal encoding.
assert df["Review Count"].ge(0).all()  # Verify that log1p will receive valid counts.
assert df.isnull().sum().sum() == 0  # Stop if any missing cells remain after cleaning.
print("Missing cells after cleaning:", df.isnull().sum().sum())  # Confirm the remaining missing-cell count.
print("Cleaned shape:", df.shape)  # Report the actual number of retained rows and columns.

Duplicates removed: 2


Missing-essential-field rows removed: 157
Missing cells after cleaning: 0
Cleaned shape: (21055, 9)


## 5. Reviewer Name — Frequency Encoding

**Why this method?** Names are nominal and have many distinct values. One-hot encoding would create too many columns; assigning increasing labels could imply an order. Frequency encoding replaces each name with its proportion of rows in the cleaned dataset.

**How it works:** `value_counts(normalize=True)` calculates proportions; `map` replaces each name by its proportion. The resulting feature describes **name frequency**, not a guaranteed number of reviews by a unique person: different people may share a name. Different names with equal frequency also receive equal values.

In [5]:
freq = df["Reviewer Name"].value_counts(normalize=True)  # Calculate the fraction of rows for each name.
enc_name = df["Reviewer Name"].map(freq).rename("Reviewer Name_freq")  # Replace names by their frequencies.
print("Distinct reviewer names:", len(freq))  # Show why a compact encoding is useful.
display(enc_name.head().to_frame())  # Preview encoded values without displaying original names.

Distinct reviewer names: 18375


,Reviewer Name_freq
0,0.000047
1,0.000047
2,0.000047
3,0.000047
4,0.000047


## 6. Profile Link — Label Encoding

**Why this method?** The section treats Profile Link as an identifier. `pd.factorize` assigns one integer to each distinct link, using first-occurrence order. The codes are compact and preserve identity, but they do **not** represent rank, distance, or quality.

**Important:** include this encoding to answer the lab; normally remove the identifier before training a predictive model. Frequency encoding would collapse many unique links to the same value, while one-hot encoding would be unnecessarily wide.

In [6]:
codes, uniques = pd.factorize(df["Profile Link"])  # Assign each distinct profile link a code.
enc_link = pd.Series(codes, index=df.index, name="Profile Link_label")  # Keep codes aligned with the original rows.
print("Distinct profile identifiers:", len(uniques))  # Report identifier cardinality without exposing the links.
display(enc_link.head().to_frame())  # Preview the numerical identifiers.

Distinct profile identifiers: 21055


,Profile Link_label
0,0
1,1
2,2
3,3
4,4


## 7. Country — One-Hot Encoding

**Why this method?** Country categories have no natural order. One-hot columns express category membership without suggesting that one country is numerically larger than another.

**Section setting:** retain the ten most frequent countries and combine the rest into `Other`. This limits the number of columns, at the cost of losing distinctions between rare countries. Every row must have exactly one active country indicator.

In [7]:
top = df["Country"].value_counts().nlargest(10).index  # Select the ten most frequent countries.
country = df["Country"].where(df["Country"].isin(top), "Other")  # Group countries outside the top ten.
enc_country = pd.get_dummies(country, prefix="Country", dtype=int)  # Create one integer indicator per retained category.
assert enc_country.sum(axis=1).eq(1).all()  # Confirm exactly one country category per row.
print("Country indicator columns:", enc_country.shape[1])  # Display the resulting width.
display(enc_country.head())  # Preview the one-hot output.

Country indicator columns: 11


,Country_AU,Country_CA,Country_DE,Country_DK,Country_GB,Country_IE,Country_IN,Country_IT,Country_NL,Country_Other,Country_US
0,0,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,1,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0


## 8. Review Count — Numeric Log Transformation

**Why this method?** Review Count is already numerical, so categorical encoding is not needed. As in the section, apply `log1p(x) = ln(1 + x)` to reduce the influence of very large counts while preserving their order.

**Why add one?** Zero is a valid count; `log1p(0)` is defined. This is a numerical transformation, not label encoding or scaling to a fixed interval. Compare the original and transformed summary statistics.

In [8]:
enc_count = np.log1p(df["Review Count"]).rename("Review Count_log")  # Compress large counts and support zero.
display(pd.concat([df["Review Count"].describe().rename("Original count"), enc_count.describe().rename("Log-transformed count")], axis=1))  # Compare distributions.

,Original count,Log-transformed count
count,21055.000000,21055.000000
mean,8.488720,1.641195
std,18.239247,0.949345
min,1.000000,0.693147
25%,1.000000,0.693147
50%,3.000000,1.386294
75%,8.000000,2.197225
max,633.000000,6.452049


## 9. Rating — Ordinal Encoding

**Why this method?** Ratings have a meaningful order: 1 < 2 < 3 < 4 < 5. We explicitly provide this order to `OrdinalEncoder`, producing 0, 1, 2, 3, 4 respectively.

**Read the code:** double brackets select a one-column DataFrame, which the encoder expects. `fit_transform` learns the mapping and applies it. Restoring `df.index` maintains row alignment. Ordered codes preserve rank, but do not prove equal psychological gaps between star levels.

**Modelling note:** if Rating is a future prediction target, separate this column as `y`; do not include it among input features.

In [9]:
oe = OrdinalEncoder(categories=[[1, 2, 3, 4, 5]])  # Define the correct star order explicitly.
enc_rating = pd.DataFrame(oe.fit_transform(df[["Rating"]].astype(int)), columns=["Rating_ordinal"], index=df.index)  # Apply the mapping and preserve row labels.
assert np.array_equal(enc_rating["Rating_ordinal"].to_numpy(), df["Rating"].to_numpy() - 1)  # Verify 1→0 through 5→4.
print("Mapping: 1→0, 2→1, 3→2, 4→3, 5→4")  # State the interpretation of the encoded labels.
display(enc_rating["Rating_ordinal"].value_counts().sort_index().rename("Number of reviews").to_frame())  # Count each encoded level.

Mapping: 1→0, 2→1, 3→2, 4→3, 5→4


,Number of reviews
Rating_ordinal,
0.0,13123
1.0,1227
2.0,885
3.0,1292
4.0,4528


## 10. Review Date — Year and Cyclic Month

**Why this method?** A date contains useful chronological and seasonal information. Following the section, extract the year and represent month with two features: `sin(2π × month / 12)` and `cos(2π × month / 12)`.

**Why both sine and cosine?** Together they locate a month on a circle, keeping December and January close. A plain month number places 12 and 1 far apart. Define the reusable `date_features` function once, then apply it to both date columns. This section representation intentionally omits day-of-month and time-of-day.

In [10]:
def date_features(s, name):  # Accept a datetime Series and an output-column prefix.
    out = pd.DataFrame(index=s.index)  # Create an empty result with the same row alignment.
    out[name + "_year"] = s.dt.year  # Extract the calendar year as a number.
    out[name + "_month_sin"] = np.sin(2 * np.pi * s.dt.month / 12)  # Compute the first cyclic month coordinate.
    out[name + "_month_cos"] = np.cos(2 * np.pi * s.dt.month / 12)  # Compute the second cyclic month coordinate.
    return out  # Return the three date features.
enc_rdate = date_features(df["Review Date"], "Review Date")  # Encode review timestamps with the reusable function.
display(enc_rdate.head())  # Preview the year and the two month coordinates.

,Review Date_year,Review Date_month_sin,Review Date_month_cos
0,2024,-1.0,-1.836970e-16
1,2024,-1.0,-1.836970e-16
2,2024,-1.0,-1.836970e-16
3,2024,-1.0,-1.836970e-16
4,2024,-1.0,-1.836970e-16


## 11. Review Title — TF-IDF with 20 Features

**Why this method?** Titles are free text, not a small set of categories. TF-IDF creates weighted word features: term frequency captures use within a document, and inverse document frequency reduces the relative influence of words common across documents. Scikit-learn also normalizes each nonzero document vector by default.

**Section settings:** `max_features=20` limits the vocabulary to at most 20 terms; `stop_words="english"` removes English stop words. The vocabulary cap selects terms by corpus term frequency, not by an independent semantic-importance score. `fit_transform` produces a sparse matrix; converting to a small dense table is manageable at this cap. The `title_` prefix prevents overlap with text features.

**Limitation:** the small vocabulary is a teaching choice and discards many words; TF-IDF does not understand word order or context.

In [11]:
tf_title = TfidfVectorizer(max_features=20, stop_words="english")  # Configure the section's title vocabulary.
X_title = tf_title.fit_transform(df["Review Title"])  # Learn the title vocabulary and compute TF-IDF weights.
enc_title = pd.DataFrame(X_title.toarray(), columns=tf_title.get_feature_names_out(), index=df.index).add_prefix("title_")  # Create aligned, clearly named features.
print("Title features:", enc_title.columns.tolist())  # Display the actual learned vocabulary.
print("Title matrix shape:", enc_title.shape)  # Show the number of rows and title features.
display(enc_title.head())  # Preview the numerical weights.

Title features: ['title_amazon', 'title_bad', 'title_best', 'title_company', 'title_customer', 'title_delivery', 'title_don', 'title_experience', 'title_good', 'title_great', 'title_horrible', 'title_item', 'title_love', 'title_order', 'title_ordered', 'title_prime', 'title_service', 'title_terrible', 'title_time', 'title_worst']
Title matrix shape: (21055, 20)


,title_amazon,title_bad,title_best,title_company,title_customer,title_delivery,title_don,title_experience,title_good,title_great,title_horrible,title_item,title_love,title_order,title_ordered,title_prime,title_service,title_terrible,title_time,title_worst
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 12. Review Text — TF-IDF with 100 Features

**Why this method?** Review Text is longer and more varied than a title, so the section uses the same TF-IDF method with up to 100 terms. A separate vectorizer learns a separate vocabulary for this column.

**Missing text:** the empty strings created during cleaning have all-zero TF-IDF vectors. Other nonempty texts may also become zero vectors if they contain no retained vocabulary terms. The code reports both counts rather than assuming they are identical.

In [12]:
tf_text = TfidfVectorizer(max_features=100, stop_words="english")  # Configure the larger review-text vocabulary.
X_text = tf_text.fit_transform(df["Review Text"])  # Learn review-text terms and calculate their weights.
enc_text = pd.DataFrame(X_text.toarray(), columns=tf_text.get_feature_names_out(), index=df.index).add_prefix("text_")  # Convert to aligned numerical features.
print("Review text matrix shape:", enc_text.shape)  # Report the real feature count.
print("Empty review texts:", df["Review Text"].eq("").sum())  # Count empty documents after cleaning.
print("All-zero text vectors:", enc_text.eq(0).all(axis=1).sum())  # Include nonempty documents with no retained terms.
display(enc_text.iloc[:5, :10])  # Preview five rows and the first ten text features.

Review text matrix shape: (21055, 100)
Empty review texts: 630
All-zero text vectors: 931


,text_account,text_address,text_amazon,text_asked,text_bad,text_bought,text_business,text_buy,text_called,text_cancel
0,0.213067,0.0,0.000000,0.0,0.0,0.000000,0.0,0.239272,0.0,0.00000
1,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.00000
2,0.321654,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.00000
3,0.000000,0.0,0.196262,0.0,0.0,0.265007,0.0,0.237637,0.0,0.00000
4,0.000000,0.0,0.215039,0.0,0.0,0.000000,0.0,0.000000,0.0,0.20935


## 13. Date of Experience — Date Features and Review Delay

**Why this method?** Reuse the section's year and cyclic-month features. Add `days_to_review`, the difference between Review Date and Date of Experience in whole days. This captures elapsed time that is not represented by the separate month/year features alone.

**Data-quality check:** report any negative delays. They may indicate inconsistent source dates and should be investigated, not silently rewritten. They are retained here to follow the lab's transformation faithfully. `.dt.days` returns whole-day components of the timedeltas; the experience date is represented at midnight UTC.

In [13]:
enc_edate = date_features(df["Date of Experience"], "Experience")  # Reuse the date-feature function.
enc_edate["days_to_review"] = (df["Review Date"] - df["Date of Experience"]).dt.days  # Calculate the whole-day review delay.
display(enc_edate["days_to_review"].describe().to_frame())  # Summarize the delay distribution.
print("Negative delays retained for review:", enc_edate["days_to_review"].lt(0).sum())  # Flag questionable dates explicitly.
display(enc_edate.head())  # Preview the four experience-related features.

,days_to_review
count,21055.000000
mean,6.244882
std,52.482829
min,-1.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,2428.000000


Negative delays retained for review: 18


,Experience_year,Experience_month_sin,Experience_month_cos,days_to_review
0,2024,-1.0,-1.836970e-16,0
1,2024,-1.0,-1.836970e-16,0
2,2024,-1.0,-1.836970e-16,0
3,2024,-1.0,-1.836970e-16,0
4,2024,-1.0,-1.836970e-16,0


## 14. Combine All Encoded Columns and Verify the Result

**What and why:** `pd.concat(..., axis=1)` joins the nine encoding blocks side by side. Shared row indexes ensure that each row still describes the same original review. The coverage table maps **every original column** to its output block and counts the resulting features.

**Checks:** confirm complete column coverage, matching row indexes, no duplicate output-column names, numeric types only, no missing cells, and no infinite values. These checks validate the encoding output, not the quality of a trained model: this assignment does not train one.

In [14]:
blocks = {"Reviewer Name": enc_name, "Profile Link": enc_link, "Country": enc_country, "Review Count": enc_count, "Rating": enc_rating, "Review Date": enc_rdate, "Review Title": enc_title, "Review Text": enc_text, "Date of Experience": enc_edate}  # Map all source fields to their encoding blocks.
assert set(blocks) == set(original_columns)  # Confirm that none of the original columns is missing.
assert all(block.index.equals(df.index) for block in blocks.values())  # Prevent row misalignment before combining.
encoded = pd.concat([enc_name, enc_link, enc_country, enc_count, enc_rating, enc_rdate, enc_edate, enc_title, enc_text], axis=1)  # Join all encoded columns horizontally.
coverage = pd.DataFrame({"Original column": list(blocks), "Output features": [1 if isinstance(block, pd.Series) else block.shape[1] for block in blocks.values()]})  # Count features per original column.
assert len(encoded) == len(df)  # Confirm that combining blocks preserved all cleaned rows.
assert encoded.columns.is_unique  # Ensure there are no duplicate feature names.
assert encoded.select_dtypes(exclude="number").empty  # Require all features to have numeric dtypes.
assert encoded.isnull().sum().sum() == 0  # Require no missing output cells.
assert np.isfinite(encoded.to_numpy(dtype=float)).all()  # Require finite numeric values throughout.
assert coverage["Output features"].sum() == encoded.shape[1]  # Verify the feature-count accounting.
display(coverage)  # Show explicit coverage of all nine original fields.
print("Final encoded shape:", encoded.shape)  # Report the actual final matrix dimensions.
print("Missing cells:", encoded.isnull().sum().sum())  # Report remaining missing output cells.
print("Numeric dtypes:")  # Label the following type summary.
print(encoded.dtypes.value_counts())  # Count the final integer and float columns.
print("All validation checks passed.")  # Reached only after every assertion succeeds.
display(encoded.iloc[:5, :15])  # Preview a readable subset of the final numeric matrix.

,Original column,Output features
0,Reviewer Name,1
1,Profile Link,1
2,Country,11
3,Review Count,1
4,Rating,1
5,Review Date,3
6,Review Title,20
7,Review Text,100
8,Date of Experience,4


Final encoded shape: (21055, 142)
Missing cells: 0
Numeric dtypes:
float64    127
int64       13
int32        2
Name: count, dtype: int64
All validation checks passed.


,Reviewer Name_freq,Profile Link_label,Country_AU,Country_CA,Country_DE,Country_DK,Country_GB,Country_IE,Country_IN,Country_IT,Country_NL,Country_Other,Country_US,Review Count_log,Rating_ordinal
0,0.000047,0,0,0,0,0,0,0,0,0,0,0,1,0.693147,0.0
1,0.000047,1,0,0,0,0,1,0,0,0,0,0,0,2.302585,0.0
2,0.000047,2,0,0,0,0,1,0,0,0,0,0,0,4.510860,0.0
3,0.000047,3,1,0,0,0,0,0,0,0,0,0,0,1.791759,0.0
4,0.000047,4,0,0,0,0,1,0,0,0,0,0,0,2.197225,0.0


## Conclusion and Video Explanation Guide
All nine original columns have been handled using the methods taught in the section. Frequency encoding handles names, label codes represent profile identifiers, one-hot encoding handles countries, log transformation handles counts, ordinal encoding preserves rating order, date features represent timing, and TF-IDF represents the two text fields. The final cell displays the real executed results and validates the combined numerical table.

For your recording, explain the notebook in order:
1. State the goal and explain why non-numeric fields need suitable representations.
2. Explain each import, the original dataset download, and the inspection outputs.
3. Explain the cleaning decisions and why types are converted before imputing values.
4. For each of the nine fields, state its type, justify the method, and explain each code line using its comment.
5. Show the actual outputs, including the delay warning and final feature-count table.
6. Explain that `axis=1` joins columns and that matching indexes preserve the rows.

### Limits and future modelling
- This is an encoding lab, **not** a trained prediction pipeline.
- Before future model evaluation, split training/test data **first**. Learn frequencies, top countries, imputations, and TF-IDF vocabularies only from training data, then transform held-out data with those fitted rules. Fitting on the full dataset here completes the lab demonstration; reusing it for test evaluation would leak information.
- Drop `Profile Link_label` from predictive inputs because it is an arbitrary identifier. If predicting Rating, separate `Rating_ordinal` as the target rather than including it among predictors.
- Grouping rare countries, short TF-IDF vocabularies, and date imputation are explicit trade-offs inherited from the section. Numerical completeness does not remove all source-data limitations.